In [1]:
#pip uninstall xgboost -y

In [2]:
#!pip install textblob xgboost scikit-learn pandas textblob gensim

In [3]:
# import nltk
# nltk.download('stopwords')

In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [5]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score
import numpy as np
import xgboost as xgb
import json
import pandas as pd
from pandas import json_normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from textblob import TextBlob
import re
from sklearn.ensemble import RandomForestClassifier
from gensim.models import Word2Vec
from gensim.models import Word2Vec


In [6]:
# ===============================
# JSONL LOADING
# ===============================


# Load the training data from a JSON Lines file (one JSON object per line)
train_data = pd.read_json('train.jsonl', lines=True)
# The tweet data is nested. json_normalize flattens the nested JSON into columns.
train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
# Also normalize the Kaggle data
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))


# Separate features from the target variable for the training set
X_train = train_data.drop('label', axis=1)
y_train = train_data['label']

X_kaggle = kaggle_data

print("✓ Chargement OK")

✓ Chargement OK


In [7]:
import pandas as pd
import numpy as np
from datetime import datetime

def create_advanced_features(df_input):
    df = df_input.copy()
    
    # Définition des séries de fallback (robustesse contre les colonnes manquantes)
    default_int_series = pd.Series(0, index=df.index)
    default_bool_series = pd.Series(False, index=df.index)
    
    # --- Initialisation des colonnes numériques ---
    df['user.followers_count'] = df.get('user.followers_count', default_int_series).fillna(0)
    df['user.friends_count'] = df.get('user.friends_count', default_int_series).fillna(0)
    df['user.listed_count'] = df.get('user.listed_count', default_int_series).fillna(0)
    df['user.favourites_count'] = df.get('user.favourites_count', default_int_series).fillna(0)
    df['user.statuses_count'] = df.get('user.statuses_count', default_int_series).fillna(0)
    df['retweet_count'] = df.get('retweet_count', default_int_series).fillna(0)
    df['favorite_count'] = df.get('favorite_count', default_int_series).fillna(0)
    df['quote_count'] = df.get('quote_count', default_int_series).fillna(0) # Nouveau : Quote Count
    df['reply_count'] = df.get('reply_count', default_int_series).fillna(0) # Nouveau : Reply Count
    
    # --- A. GESTION DES DATES (Ancienneté du compte) ---
    df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
    ref_date = pd.to_datetime('now', utc=True)
    df['account_age_days'] = (ref_date - df['user_created_at_dt']).dt.days
    df['account_age_days'] = df['account_age_days'].fillna(0)
    
    # Extraction des indicateurs temporels avancés du tweet
    df['created_at_dt'] = pd.to_datetime(df.get('created_at'), errors='coerce')
    df['tweet_hour'] = df['created_at_dt'].dt.hour.fillna(-1)
    df['tweet_is_weekend'] = df['created_at_dt'].dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    # --- B. QUALITÉ DU PROFIL & STATUT (Booléens) ---
    df['is_default_profile'] = df.get('user.default_profile', default_bool_series).fillna(False).astype(int)
    df['is_default_image'] = df.get('user.default_profile_image', default_bool_series).fillna(False).astype(int)
    df['is_verified'] = df.get('user.verified', default_bool_series).fillna(False).astype(int)
    
    # Nouveau : Est-ce un compte protégé/privé ? (Signe d'un follower ou d'un utilisateur personnel)
    df['is_protected'] = df.get('user.protected', default_bool_series).fillna(False).astype(int)
    
    # Nouveau : Le profil a-t-il une URL renseignée ?
    df['has_url'] = df.get('user.url', pd.Series(False, index=df.index)).notna().astype(int)

    # --- C. CONTENU DU TWEET (Entities Counting) ---
    def count_entities(x):
        if isinstance(x, list) or (isinstance(x, pd.Series) and x.dtype == object): return len(x)
        return 0

    df['num_urls'] = df.get('entities.urls', default_int_series).apply(count_entities)
    df['num_hashtags'] = df.get('entities.hashtags', default_int_series).apply(count_entities)
    df['num_mentions'] = df.get('entities.user_mentions', default_int_series).apply(count_entities)
    df['has_media'] = df.get('extended_entities.media', default_bool_series).notna().astype(int)

    # --- D. RATIOS PUISSANTS & COMPORTEMENTAUX ---
    followers = df['user.followers_count']
    friends = df['user.friends_count']
    listed = df['user.listed_count']
    statuses = df['user.statuses_count']
    
    # 1. Ratio Followers / Friends (Ratio de notoriété)
    df['ratio_followers_friends'] = followers / (friends + 1)
    df['ratio_listed_followers'] = listed / (followers + 1)
    
    # 2. Taux de Réciprocité d'Amitié (Nouveau - Indique un équilibre/déséquilibre d'influence)
    df['reciprocity_score'] = (friends - followers) / (friends + followers + 1)

    # 3. Activité (Tweets par jour d'existence)
    df['tweets_per_day'] = statuses / (df['account_age_days'] + 1)
    
    # 4. Ratio Mention/Tweet (Nouveau - Taux d'interaction vs. diffusion)
    # Plus ce ratio est élevé, plus l'utilisateur interagit personnellement (follower).
    df['ratio_mention_status'] = df['num_mentions'] / (statuses + 1)
    
    # 5. Engagement (Taux d'engagement par Tweet)
    total_engagement = df['retweet_count'] + df['favorite_count'] + df['quote_count'] + df['reply_count']
    df['total_tweet_engagement'] = total_engagement / (followers + 1)

    # --- E. LONGUEUR DES TEXTES ---
    df['final_text'] = df.get('extended_tweet.full_text', df.get('text', pd.Series(''))).fillna('')
    df['final_text'] = df['final_text'].where(df['final_text'] != '', df.get('text', '')).fillna('')
    
    df['text_length'] = df['final_text'].astype(str).apply(len)
    df['bio_length'] = df.get('user.description', '').astype(str).apply(len)

    # --- F. SÉLECTION FINALE ---
    features_to_keep = [
        # Métriques User Brutes
        'user.followers_count', 'user.friends_count', 'user.listed_count', 
        'user.favourites_count', 'user.statuses_count',
        # Métriques Tweet Brutes
        'retweet_count', 'favorite_count', 'quote_count', 'reply_count',
        # Ratios & Comportementaux (NOUVEAU)
        'ratio_followers_friends', 'ratio_listed_followers', 'tweets_per_day', 'account_age_days',
        'reciprocity_score', 'ratio_mention_status', 'total_tweet_engagement',
        # Booléens & Qualité (MIS À JOUR)
        'is_verified', 'is_default_profile', 'is_default_image', 'is_geo_enabled',
        'is_protected', 'has_url',
        # Temporels (NOUVEAU)
        'tweet_hour', 'tweet_is_weekend',
        # Longueur du Contenu
        'text_length', 'bio_length', 
        # Compte des Entités
        'num_urls', 'num_hashtags', 'num_mentions', 'has_media',
    ]
    
    final_cols = [c for c in features_to_keep if c in df.columns]
    
    return df[final_cols].fillna(0)



In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from textblob import TextBlob
import re

def create_nlp_features(df_train, df_test, y_train):
    """
    Crée des features NLP (TF-IDF des Bios et Sentiment des Tweets).

    Args:
        df_train (pd.DataFrame): DataFrame d'entraînement complet (X_train).
        df_test (pd.DataFrame): DataFrame de test (X_kaggle).
        y_train (pd.Series): Cible d'entraînement (y_train).

    Returns:
        tuple: (df_train_nlp, df_test_nlp) avec les nouvelles colonnes.
    """
    
    # ----------------------------------------
    # Préparation du texte
    # ----------------------------------------
    
    # Remplacer les NaN ou valeurs manquantes par une chaîne vide
    train_bio = df_train.get('user.description', pd.Series([''] * len(df_train))).fillna('').astype(str)
    test_bio = df_test.get('user.description', pd.Series([''] * len(df_test))).fillna('').astype(str)

    # Récupération du 'final_text' du tweet (le plus complet)
    # Note: On doit reproduire la logique de 'final_text' de la fonction d'ingénierie
    def get_final_text(df):
        text = df.get('text', pd.Series([''] * len(df))).fillna('')
        full_text = df.get('extended_tweet.full_text', text).fillna(text)
        return full_text.astype(str)
        
    train_text = get_final_text(df_train)
    test_text = get_final_text(df_test)

    # ----------------------------------------
    # A. TF-IDF sur les tweets (Meta-Feature)
    # ----------------------------------------
    print("TF-IDF vectorization du corps du tweet")

    # Nettoyage très basique du texte pour le TF-IDF
    def clean_text(text):
        #text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Suppression des URLs
        #text = re.sub(r'[^\w\s]', '', text) # Suppression de la ponctuation
        return text

    train_text_clean = train_text.apply(clean_text)
    test_text_clean = test_text.apply(clean_text)

    # 1. Création du Vecteur TF-IDF (Appris uniquement sur le training set)
    tfidf = TfidfVectorizer(max_features=1000, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf_tweet = tfidf.fit_transform(train_text_clean)
    X_test_tfidf_tweet = tfidf.transform(test_text_clean)

    # 2. Entraînement du Méta-Modèle (Régression Logistique)
    log_reg = LogisticRegression(solver='sag', random_state=42)
    log_reg.fit(X_train_tfidf_tweet, y_train.astype(int))

    # 3. Extraction de la probabilité prédite (Meta-Feature)
    # Nous utilisons la probabilité pour la classe 1 (Influencer)
    train_tweet_proba = log_reg.predict_proba(X_train_tfidf_tweet)[:, 1]
    test_tweet_proba = log_reg.predict_proba(X_test_tfidf_tweet)[:, 1]    

    # ----------------------------------------
    # B. TF-IDF sur les Bios (Meta-Feature)
    # ----------------------------------------
    print("  -> Calcul du TF-IDF sur les Bios et entraînement du Méta-Modèle...")
    

    train_bio_clean = train_bio.apply(clean_text)
    test_bio_clean = test_bio.apply(clean_text)

    # 1. Création du Vecteur TF-IDF (Appris uniquement sur le training set)
    tfidf = TfidfVectorizer(max_features=1000, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf = tfidf.fit_transform(train_bio_clean)
    X_test_tfidf = tfidf.transform(test_bio_clean)

    # 2. Entraînement du Méta-Modèle (Régression Logistique)
    log_reg = LogisticRegression(solver='liblinear', random_state=42)
    log_reg.fit(X_train_tfidf, y_train.astype(int))

    # 3. Extraction de la probabilité prédite (Meta-Feature)
    # Nous utilisons la probabilité pour la classe 1 (Influencer)
    train_bio_proba = log_reg.predict_proba(X_train_tfidf)[:, 1]
    test_bio_proba = log_reg.predict_proba(X_test_tfidf)[:, 1]

    # ----------------------------------------
    # B. Analyse du Sentiment (Polarity et Subjectivity)
    # ----------------------------------------
    print("  -> Extraction du Sentiment (Polarity/Subjectivity) des Tweets...")
    
    # La fonction TextBlob est utilisée pour obtenir les scores de sentiment
    # C'est une opération lente, il faut être patient
    def get_sentiment(text):
        try:
            analysis = TextBlob(text)
            return pd.Series({'polarity': analysis.sentiment.polarity, 'subjectivity': analysis.sentiment.subjectivity})
        except:
            return pd.Series({'polarity': 0.0, 'subjectivity': 0.0})

    train_sentiment = train_text.apply(get_sentiment)
    test_sentiment = test_text.apply(get_sentiment)

    # ----------------------------------------
    # 4. Fusion des Features NLP
    # ----------------------------------------
    
    # Création des DataFrames de features NLP
    df_train_nlp = pd.DataFrame({
        'meta_bio_proba': train_bio_proba,
        'tweet_polarity': train_sentiment['polarity'],
        'tweet_subjectivity': train_sentiment['subjectivity'],
        'meta_tweet_proba': train_tweet_proba
    })
    
    df_test_nlp = pd.DataFrame({
        'meta_bio_proba': test_bio_proba,
        'tweet_polarity': test_sentiment['polarity'],
        'tweet_subjectivity': test_sentiment['subjectivity'],
        'meta_tweet_proba': test_tweet_proba
    })

    return df_train_nlp, df_test_nlp

In [9]:
# =======================================================
# On concatène ca dans X_train_advanced et X_kaggle_advanced
# =======================================================
print("🛠️ Construction des features avancées (Métadonnées)...")

# Application de la fonction robuste (Métadonnées)
X_train_advanced = create_advanced_features(X_train)
X_kaggle_advanced = create_advanced_features(X_kaggle)

# Préparation de la cible
y_train_clean = y_train.astype(int)

# --- NOUVELLE ÉTAPE : CRÉATION DES FEATURES NLP ---
X_train_nlp, X_kaggle_nlp = create_nlp_features(X_train, X_kaggle, y_train_clean)

# --- FUSION DES FEATURES ---
X_train_advanced = pd.concat([X_train_advanced, X_train_nlp], axis=1)
X_kaggle_advanced = pd.concat([X_kaggle_advanced, X_kaggle_nlp], axis=1)

print(f"\nFeatures combinées ({len(X_train_advanced.columns)}):")
print(list(X_train_advanced.columns))

🛠️ Construction des features avancées (Métadonnées)...


/var/folders/cw/21kvxr9n5h78kj_tr08w1c2w0000gn/T/ipykernel_11813/3318943902.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
/var/folders/cw/21kvxr9n5h78kj_tr08w1c2w0000gn/T/ipykernel_11813/3318943902.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')


TF-IDF vectorization du corps du tweet
  -> Calcul du TF-IDF sur les Bios et entraînement du Méta-Modèle...
  -> Extraction du Sentiment (Polarity/Subjectivity) des Tweets...

Features combinées (33):
['user.followers_count', 'user.friends_count', 'user.listed_count', 'user.favourites_count', 'user.statuses_count', 'retweet_count', 'favorite_count', 'quote_count', 'reply_count', 'ratio_followers_friends', 'ratio_listed_followers', 'tweets_per_day', 'account_age_days', 'reciprocity_score', 'ratio_mention_status', 'total_tweet_engagement', 'is_verified', 'is_default_profile', 'is_default_image', 'is_protected', 'has_url', 'tweet_hour', 'tweet_is_weekend', 'text_length', 'bio_length', 'num_urls', 'num_hashtags', 'num_mentions', 'has_media', 'meta_bio_proba', 'tweet_polarity', 'tweet_subjectivity', 'meta_tweet_proba']


On ajoute les embedding par Word2vec

In [10]:
def simple_tokenize(text):
    """
    Tokenise un texte en le mettant en minuscules et en le séparant par espace.
    Nettoie les caractères non alphanumériques courants pour Word2Vec.
    """
    # 1. Mise en minuscule
    text = text.lower()
    
    # 2. Remplacer la ponctuation courante par un espace pour la séparation
    # Ceci est une simplification pour Word2Vec
    text = re.sub(r'[.,;:`"\'!?()]', ' ', text)
    
    # 3. Séparation par espace et suppression des entrées vides
    return [word for word in text.split() if word]

# ====================================================================
# 2. CHARGEMENT ET PRÉPARATION DES DONNÉES (Votre Bloc)
# ====================================================================

# --- 1. FONCTION D'EXTRACTION DE TEXTE ---
def extract_full_text(row):
    """Extrait le texte le plus complet disponible du tweet."""
    if "extended_tweet.full_text" in row and pd.notna(row["extended_tweet.full_text"]):
        return row["extended_tweet.full_text"]
    if "text" in row and pd.notna(row["text"]):
        return row["text"]
    return ""

# --- 2. CHARGEMENT DES DONNÉES BRUTES ---
try:
    # 🚨 Adaptez les chemins de fichiers si nécessaire !
    # Application de l'extraction de texte
    train_data["full_text"] = train_data.apply(extract_full_text, axis=1)
    kaggle_data["full_text"] = kaggle_data.apply(extract_full_text, axis=1)

    # DÉFINITION DES VARIABLES POUR LE PIPELINE W2V
    X_full = train_data["full_text"].values # Texte complet d'entraînement
    y_full = train_data["label"].values.astype(int) # Labels
    X_kaggle_full = kaggle_data["full_text"].values # Texte complet Kaggle
    
    print("✓ Données brutes (X_full, y_full) chargées et prêtes.")

except FileNotFoundError as e:
    print(f"❌ ERREUR: Fichier introuvable. Vérifiez votre chemin : {e}")
    exit()

✓ Données brutes (X_full, y_full) chargées et prêtes.


In [11]:
# ====================================================================
# CONFIGURATION WORD2VEC (Assurez-vous que ces valeurs sont correctes)
# ====================================================================
EMBEDDING_DIM = 250 
WINDOW_SIZE = 5
MIN_COUNT = 1
# Note : Si vous avez entraîné votre modèle W2V sur toutes les données, 
# vous pouvez utiliser X_full. Sinon, utilisez X_train_text.
# Ici, nous utilisons les ensembles complets pour la vectorisation.

# ====================================================================
# 1. PRÉPARATION DES DONNÉES DE TEXTE (Tokenisation)
# ====================================================================

# 🚨 Assurez-vous que X_full et X_kaggle_full contiennent le texte brut !

# J'utilise ici la fonction simple_tokenize définie dans la réponse précédente
# (Elle doit être définie dans votre session pour que ce bloc fonctionne)

print("\n🔄 Tokenisation des données de texte...")
# 1. Tokenisation du jeu d'entraînement (X_full)
X_full_tokenized = [simple_tokenize(text) for text in X_full]

# 2. Tokenisation du jeu Kaggle (X_kaggle_full)
X_kaggle_tokenized = [simple_tokenize(text) for text in X_kaggle_full]


# ====================================================================
# 2. FONCTION DE VECTORISATION
# ====================================================================

def document_vectorizer(tokens, model, dim):
    """Calcule le vecteur moyen pour un document."""
    vector = np.zeros(dim)
    count = 0
    # Ne fait la moyenne que sur les mots présents dans le vocabulaire W2V
    for word in tokens:
        if word in model.wv:
            vector += model.wv[word]
            count += 1
    
    if count != 0:
        vector /= count
        
    return vector

# ====================================================================
# 3. GÉNÉRATION DES EMBEDDINGS (VECTEURS)
# ====================================================================

print("🔄 Génération des embeddings W2V par moyenne...")

# 🚨 Le modèle w2v_model doit être défini ou chargé ici
# Si vous ne l'avez pas entraîné ou chargé dans la même session, vous devez le faire ici !
# Exemple: w2v_model = Word2Vec.load("mon_modele_w2v.model")

w2v_model = Word2Vec(
    sentences=X_full_tokenized, 
    vector_size=EMBEDDING_DIM, 
    window=WINDOW_SIZE, 
    min_count=MIN_COUNT, 
    sg=1 # 1: Skip-gram (souvent meilleur pour la sémantique), 0: CBOW
)

# 3.1. Entraînement
X_train_vectors = np.array([document_vectorizer(tokens, w2v_model, EMBEDDING_DIM) for tokens in X_full_tokenized])

# 3.2. Kaggle
X_kaggle_vectors = np.array([document_vectorizer(tokens, w2v_model, EMBEDDING_DIM) for tokens in X_kaggle_tokenized])


# ====================================================================
# 4. FUSION DES EMBEDDINGS DANS X_train_advanced
# ====================================================================

print("🔄 Fusion des embeddings W2V avec les jeux X_advanced...")

# Création des DataFrames d'Embeddings
embed_cols = [f'w2v_e_{i}' for i in range(EMBEDDING_DIM)]

X_train_nlp = pd.DataFrame(X_train_vectors, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vectors, columns=embed_cols)

# Fusion des Caractéristiques (Métadonnées + Embeddings W2V)
X_train_advanced = pd.concat([X_train_advanced, X_train_nlp], axis=1)
X_kaggle_advanced = pd.concat([X_kaggle_advanced, X_kaggle_nlp], axis=1)

print("\n" + "=" * 50)
print("✓ FUSION DES EMBEDDINGS W2V TERMINÉE")
print("=" * 50)
print(f"Total Features d'entraînement (X_train_advanced) : {len(X_train_advanced.columns)}")
print(f"Total Features de test (X_kaggle_advanced) : {len(X_kaggle_advanced.columns)}")


🔄 Tokenisation des données de texte...
🔄 Génération des embeddings W2V par moyenne...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

🔄 Fusion des embeddings W2V avec les jeux X_advanced...

✓ FUSION DES EMBEDDINGS W2V TERMINÉE
Total Features d'entraînement (X_train_advanced) : 283
Total Features de test (X_kaggle_advanced) : 283


Xgboost final

In [12]:
# #=======================================================
# # 3. ENTRAÎNEMENT ET PRÉDICTION (Optimisation Mémoire)
# # =======================================================
# print("\n🔄 Lancement du Random Search pour l'optimisation XGBoost...")

# # Grille de paramètres optimisée pour la mémoire
# # {'subsample': 0.9, 'n_estimators': 300, 'max_depth': 15, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 0.8}

# xgb_model = xgb.XGBClassifier(
#     use_label_encoder=False, 
#     eval_metric='logloss', 
#     random_state=42,
#     tree_method='hist', # Reste 'hist' pour l'efficacité, mais cela ne résout pas le problème de RAM.,
#     # Paramètres issus de la cross val de 1100 minutes lol
#     colsample_bytree=0.6,
#     gamma=0.1,
#     learning_rate=0.1,
#     max_depth=7,
#     n_estimators=300,
#     subsample=0.8,
#     # On farfouille
#     reg_lambda=69
# )

# # Custom scoring to include early stopping
# scoring = make_scorer(
#     score_func=lambda y_true, y_pred: accuracy_score(y_true, y_pred),
#     greater_is_better=True,
# )

# # Create a pipeline that includes early stopping
# early_stopping_steps = 20  # Number of epochs without improvement to stop

# xgb_model.fit(X_train_advanced, y_train_clean)

# #print(f"\n🏆 Meilleure Accuracy (Cross-Validation) : {xgb_model.best_score_:.4f}")

# # Prédiction sur le set Kaggle
# y_pred_kaggle = xgb_model.predict(X_kaggle_advanced)

# # =======================================================
# # 4. GÉNÉRATION DU FICHIER DE SOUMISSION
# # =======================================================

# output = pd.concat([X_kaggle['challenge_id'], pd.DataFrame(y_pred_kaggle)], axis=1, ignore_index=True)
# output.columns = ['ID', "Prediction"]
# output.to_csv('submission_xgboost_advanced_final.csv', index=False)

# print("\n✓ Fichier 'submission_xgboost_advanced_final.csv' généré avec succès !")
# print(output.head())

In [13]:
xgb_model.__dict__

NameError: name 'xgb_model' is not defined

In [14]:
!pip install torch

In [15]:
!pip uninstall numpy -y

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4


In [16]:
!pip install numpy==1.26.4

  Using cached numpy-1.26.4-cp311-cp311-macosx_10_9_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-macosx_10_9_x86_64.whl (20.6 MB)


In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset


In [18]:
# Features and labels
X_train_tensor = torch.tensor(X_train_advanced.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_clean.values, dtype=torch.long)
print("Train set ops baby")

X_kaggle_tensor = torch.tensor(X_kaggle_advanced.values, dtype=torch.float32)
print("Kaggle set ops baby")

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_train_tensor = X_train_tensor.to(device)
y_train_tensor = y_train_tensor.to(device)
X_kaggle_tensor = X_kaggle_tensor.to(device)


Train set ops baby
Kaggle set ops baby


In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TabularTransformer(nn.Module):
    def __init__(self, input_dim, emb_dim=64, num_heads=4, num_layers=4, dropout=0.1):
        super(TabularTransformer, self).__init__()

        # Chaque feature → embedding (linear projection)
        self.feature_embedding = nn.Linear(1, emb_dim)

        # Encodeur Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            activation='relu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(input_dim * emb_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        # x shape : (batch_size, input_dim)

        # Ajouter une dimension : (batch_size, input_dim, 1)
        x = x.unsqueeze(-1)

        # Embedding par feature : (batch_size, input_dim, emb_dim)
        x = self.feature_embedding(x)

        # Transformer Encoder
        x = self.transformer(x)

        # On aplanit pour la classification
        x = x.flatten(start_dim=1)

        return self.classifier(x)


In [23]:
dataset = TensorDataset(X_train_tensor, y_train_tensor)
loader = DataLoader(dataset, batch_size=128, shuffle=True)


In [24]:
model = TabularTransformer(X_train_advanced.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 50

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(loader):.4f}")


KeyboardInterrupt: 

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X_kaggle_tensor)
    y_pred_kaggle_dl = torch.argmax(preds, dim=1).cpu().numpy()

# Save submission
output_dl = pd.DataFrame({
    'ID': X_kaggle['challenge_id'],
    'Prediction': y_pred_kaggle_dl
})
output_dl.to_csv('submission_dl_ffnn.csv', index=False)
print("✓ Submission for PyTorch model saved as 'submission_dl_ffnn.csv'")


✓ Submission for PyTorch model saved as 'submission_dl_ffnn.csv'
